In [1]:
from utils import prepare_dataframe

df = prepare_dataframe("merged.csv")
resulting_label_count = df['labels'].nunique()
df

labels
bug            17934
enhancement    16113
clean          15336
performance     5332
refactor        3320
Name: count, dtype: int64
labels
enhancement    15300
bug            15300
clean          15300
Name: count, dtype: int64


,full_repo_name,labels,patch,index
20575,apache/incubator-nemo,enhancement,"@@ -68,4 +68,9 @@ public Set<ExecutorRepresent...",20575
22175,apache/maven-invoker-plugin,enhancement,"@@ -43,12 +43,12 @@\n import java.util.List;\n...",22175
45510,apache/pulsar,enhancement,"@@ -486,8 +486,11 @@ public void testCardinali...",45510
15849,apache/hbase,enhancement,"@@ -1962,9 +1962,9 @@ private void startServic...",15849
5140,apache/cloudstack,enhancement,"@@ -1887,18 +1887,10 @@ public static HostResp...",5140
...,...,...,...,...
24020,apache/netbeans,clean,"@@ -45,7 +45,7 @@ public class WLTargetModuleI...",24020
25558,apache/netbeans,clean,"@@ -95,11 +95,10 @@ public Item getObjectAt(St...",25558
4503,apache/cloudstack,clean,"@@ -113,7 +113,7 @@ public Date getRemoved() {...",4503
27264,apache/netbeans,clean,"@@ -38,7 +38,7 @@ public class SuperPatternFil...",27264


In [2]:
embedding_directory = 'embeddings/GCB_Pooler_embeddings_patch_before_largewithcodedata_15300limit3label'

In [3]:
import pandas as pd
import torch

# One-hot encoding ile labels sütununu dönüştürme
df_encoded_labels = pd.get_dummies(df, columns=['labels'])
df_encoded_repo = pd.get_dummies(df, columns=['full_repo_name'])

# Özellikler ve etiketler
X_repo = df_encoded_repo.drop(columns=['labels','patch','index'])
X_repo = X_repo.astype(int)

# y = df_encoded.drop(columns=['content_before', 'content_after', 'tokens'])
y = df_encoded_labels.drop(columns=['patch', 'full_repo_name','index'])
y = y.astype(int) # convert it 0,1     instead of true, false 

df['label_tensor'] =[torch.tensor(row,dtype=torch.float32) for row in y.values] 
df['repo_name_tensor'] =[torch.tensor(row,dtype=torch.float32) for row in X_repo.values] 

df = df.drop(columns=['full_repo_name', 'labels','patch'])


print(df['label_tensor'][0] )
print(df['repo_name_tensor'][0] )
df

tensor([1., 0., 0.])
tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0.])


,index,label_tensor,repo_name_tensor
20575,20575,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
22175,22175,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
45510,45510,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
15849,15849,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
5140,5140,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
...,...,...,...
24020,24020,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
25558,25558,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
4503,4503,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
27264,27264,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."


In [4]:
from utils import load_embeddings_from_directory_for_df
import pandas as pd

df = load_embeddings_from_directory_for_df(directory_path=embedding_directory,df=df)


  0%|          | 0/45900 [00:00<?, ?it/s]

In [5]:
from sklearn.model_selection import train_test_split

X_index = df[['index']]  
# Split the code data into test, train and validation sets
X_index_train, X_index_test = train_test_split(X_index, test_size=0.3, random_state=42) #random_state: 0,3074,42
X_index_test, X_index_val = train_test_split(X_index_test, test_size=0.5, random_state=42)

In [6]:
import gc
import torch

df_train = df[df['index'].isin(X_index_train['index'].values)].reset_index(drop=True)
df_val = df[df['index'].isin(X_index_val['index'].values)].reset_index(drop=True)
df_test = df[df['index'].isin(X_index_test['index'].values)].reset_index(drop=True)

torch.cuda.empty_cache()

df_train

,index,label_tensor,repo_name_tensor,embedding
0,45510,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0....","[tensor(-0.0029), tensor(0.0908), tensor(0.482..."
1,15849,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0....","[tensor(-0.0856), tensor(0.0825), tensor(0.423..."
2,46694,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0....","[tensor(-0.0422), tensor(-0.0916), tensor(0.47..."
3,2904,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0....","[tensor(0.0331), tensor(0.0548), tensor(0.6226..."
4,54362,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0....","[tensor(0.0202), tensor(0.0028), tensor(0.4435..."
...,...,...,...,...
32125,24020,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0....","[tensor(0.0980), tensor(0.0561), tensor(0.5557..."
32126,25558,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0....","[tensor(-0.1340), tensor(0.1632), tensor(0.522..."
32127,4503,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0....","[tensor(0.0082), tensor(-0.0061), tensor(0.522..."
32128,27264,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0....","[tensor(-0.0277), tensor(0.0247), tensor(0.507..."


In [7]:
from torch.utils.data import DataLoader, Dataset
from utils import load_embedding_from_directory

# Custom Dataset Class to handle the dataframe
class CodeDataset(Dataset):
    def __init__(self, df):
        self.df = df
        self.id_list = df['index'].values 

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
    
        index = self.id_list[idx]  

        tensor_3d = self.df.loc[self.df['index'] == index, 'embedding'].iloc[0]
        repo_name_tensor = self.df.loc[self.df['index'] == index, 'repo_name_tensor'].iloc[0]
        label_tensor = self.df.loc[self.df['index'] == index, 'label_tensor'].iloc[0]
        
        #print(f"{repo_name_tensor.shape} {tensor_3d.shape} {label_tensor.shape}")
        #print("-------------------------------")
        
        #Flattening
        #embedding = torch.flatten(tensor_3d)
        embedding = tensor_3d
        return embedding, label_tensor, repo_name_tensor, index


In [8]:
# Create DataLoader

# Creating a DataLoader
def create_dataloader(df, batch_size=32, shuffle=True):
    dataset = CodeDataset(df)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

batch_size = 128
test_loader = create_dataloader(df=df_test, batch_size=batch_size)
train_loader = create_dataloader(df=df_train, batch_size=batch_size)
val_loader = create_dataloader(df=df_val,batch_size=batch_size)

full_loader_for_scaler = create_dataloader(df=df, batch_size=512)

In [9]:
from sklearn.preprocessing import StandardScaler
import tqdm

scaler = StandardScaler()

for batch_X, batch_y, batch_X_repo, idx in tqdm.tqdm_notebook(full_loader_for_scaler):
    scaler.partial_fit(batch_X)

C:\Users\FSM\AppData\Local\Temp\ipykernel_51976\796712526.py:6: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for batch_X, batch_y, batch_X_repo, idx in tqdm.tqdm_notebook(full_loader_for_scaler):


  0%|          | 0/90 [00:00<?, ?it/s]

In [10]:
import joblib

# Save scaler into file
joblib.dump(scaler, 'scalers/scaler_pooler_3labels_15300.pkl')

['scalers/scaler_pooler_3labels_15300.pkl']

In [9]:
import joblib

# Load scaler from file
scaler = joblib.load('scalers/scaler_3labels_15300.pkl')

FileNotFoundError: [Errno 2] No such file or directory: 'scalers/scaler_3labels_15300.pkl'

In [ ]:
import torch.nn as nn

# Define the ANN model
class SinglePipelineCodeDiffNN(nn.Module):
    def __init__(self, code_input_size, hidden_sizes, output_size, dropout_rate=0.05):
        super(SinglePipelineCodeDiffNN, self).__init__()
        
        # Pipeline for code embeddings
        self.code_fc1 = nn.Linear(code_input_size, hidden_sizes[0])
        self.code_relu1 = nn.ReLU()
        self.code_dropout1 = nn.Dropout(dropout_rate)
        
        self.code_fc2 = nn.Linear(hidden_sizes[0], hidden_sizes[1])
        self.code_relu2 = nn.ReLU()
        self.code_dropout2 = nn.Dropout(dropout_rate)

        self.code_fc3 = nn.Linear(hidden_sizes[1], hidden_sizes[2])
        self.code_relu3 = nn.ReLU()
        self.code_dropout3 = nn.Dropout(dropout_rate)

        self.code_fc4 = nn.Linear(hidden_sizes[2], hidden_sizes[3])
        self.code_relu4 = nn.ReLU()
        self.code_dropout4 = nn.Dropout(dropout_rate)

        self.code_fc5 = nn.Linear(hidden_sizes[3], hidden_sizes[4])
        self.code_relu5 = nn.ReLU()
        
        self.code_fc_out = nn.Linear(hidden_sizes[4], output_size)

        
    def forward(self, code_x):
        # Process code embeddings
        code_x = self.code_fc1(code_x)
        code_x = self.code_relu1(code_x)
        code_x = self.code_dropout1(code_x)
        
        code_x = self.code_fc2(code_x)
        code_x = self.code_relu2(code_x)
        code_x = self.code_dropout2(code_x)

        code_x = self.code_fc3(code_x)
        code_x = self.code_relu3(code_x)
        code_x = self.code_dropout3(code_x)

        code_x = self.code_fc4(code_x)
        code_x = self.code_relu4(code_x)
        code_x = self.code_dropout4(code_x)

        code_x = self.code_fc5(code_x)
        code_x = self.code_relu5(code_x)

        # Final output
        output = self.code_fc_out(code_x)
        return output


In [ ]:
import torch.nn as nn

# Define the ANN model
class CodeDiffNN(nn.Module):
    def __init__(self, code_input_size, repo_input_size, hidden_sizes_pipeline_1,hidden_sizes_pipeline_2, output_size, dropout_rate=0.2):
        super(CodeDiffNN, self).__init__()
        
        # Pipeline for code embeddings
        self.code_fc1 = nn.Linear(code_input_size, hidden_sizes_pipeline_1[0])
        self.code_relu1 = nn.ReLU()
        self.code_dropout1 = nn.Dropout(dropout_rate)
        
        self.code_fc2 = nn.Linear(hidden_sizes_pipeline_1[0], hidden_sizes_pipeline_1[1])
        self.code_relu2 = nn.ReLU()
        self.code_dropout2 = nn.Dropout(dropout_rate)
        
        self.code_fc3 = nn.Linear(hidden_sizes_pipeline_1[1], hidden_sizes_pipeline_1[2])
        self.code_relu3 = nn.ReLU()
        self.code_dropout3 = nn.Dropout(dropout_rate)  
        
        
        # Pipeline for repo names
        self.repo_fc1 = nn.Linear(repo_input_size, hidden_sizes_pipeline_2[0])
        self.repo_relu1 = nn.ReLU()
        self.repo_dropout1 = nn.Dropout(dropout_rate)
        
        self.repo_fc2 = nn.Linear(hidden_sizes_pipeline_2[0], hidden_sizes_pipeline_2[1])
        self.repo_relu2 = nn.ReLU()
        self.repo_dropout2 = nn.Dropout(dropout_rate)

        
        # Combining the outputs of both pipelines
        combined_input_size = hidden_sizes_pipeline_1[2] + hidden_sizes_pipeline_2[1]   # Since we're combining two pipelines
        self.fc_combined1 = nn.Linear(combined_input_size, hidden_sizes_pipeline_2[2])
        self.relu_combined1 = nn.ReLU()
        self.dropout_combined1 = nn.Dropout(dropout_rate)

        self.fc_combined2 = nn.Linear(hidden_sizes_pipeline_2[2], hidden_sizes_pipeline_2[3])
        self.relu_combined2 = nn.ReLU()
        self.dropout_combined2 = nn.Dropout(dropout_rate)

        # Output layer
        self.fc_out = nn.Linear(hidden_sizes_pipeline_2[3], output_size)
        
    def forward(self, code_x, repo_x):
        # Process code embeddings
        code_x = self.code_fc1(code_x)
        code_x = self.code_relu1(code_x)
        code_x = self.code_dropout1(code_x)
        
        code_x = self.code_fc2(code_x)
        code_x = self.code_relu2(code_x)
        code_x = self.code_dropout2(code_x)
        
        code_x = self.code_fc3(code_x)
        code_x = self.code_relu3(code_x)
        code_x = self.code_dropout3(code_x)
        
        # Process repo names
        repo_x = self.repo_fc1(repo_x)
        repo_x = self.repo_relu1(repo_x)
        repo_x = self.repo_dropout1(repo_x)
        
        repo_x = self.repo_fc2(repo_x)
        repo_x = self.repo_relu2(repo_x)
        repo_x = self.repo_dropout2(repo_x)
        
        # Combine both outputs
        combined_x = torch.cat((code_x, repo_x), dim=1)
        
        # Process the combined output
        combined_x = self.fc_combined1(combined_x)
        combined_x = self.relu_combined1(combined_x)
        combined_x = self.dropout_combined1(combined_x)

        combined_x = self.fc_combined2(combined_x)
        combined_x = self.relu_combined2(combined_x)
        combined_x = self.dropout_combined2(combined_x)
        
        # Final output
        output = self.fc_out(combined_x)
        return output


In [ ]:
# import torch.nn as nn

# # Define the ANN model
# class CodeDiffNN(nn.Module):
#     def __init__(self, code_embedding_dim, repo_input_size, hidden_sizes, output_size, dropout_rate=0.001):
#         super(CodeDiffNN, self).__init__()
        
#         # Pipeline for code embeddings (Conv2D for 2D input)
#         # Set in_channels to the embedding dimension (e.g., 768)
#         # Increase out_channels to better handle high-dimensional inputs
#         self.code_conv1 = nn.Conv2d(in_channels=code_embedding_dim, out_channels=128, kernel_size=3, padding=1)
#         self.code_relu1 = nn.ReLU()
#         self.code_dropout1 = nn.Dropout(dropout_rate)
        
#         self.code_conv2 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
#         self.code_relu2 = nn.ReLU()
#         self.code_dropout2 = nn.Dropout(dropout_rate)

#         # Assuming a fixed input size for flattening (adjust this based on the input size and conv layer operations)
#         self.code_flatten = nn.Flatten()

#         # Flattened output from conv layers should map to hidden_sizes[1]
#         self.code_fc = nn.Linear(256*512, hidden_sizes[1])  # Modify based on input dimensions
        
#         # Pipeline for repo names (still using Linear layers for 1D input)
#         self.repo_fc1 = nn.Linear(repo_input_size, hidden_sizes[0])
#         self.repo_relu1 = nn.ReLU()
#         self.repo_dropout1 = nn.Dropout(dropout_rate)
        
#         self.repo_fc2 = nn.Linear(hidden_sizes[0], hidden_sizes[1])
#         self.repo_relu2 = nn.ReLU()
#         self.repo_dropout2 = nn.Dropout(dropout_rate)
        
#         # Combining the outputs of both pipelines
#         combined_input_size = hidden_sizes[1] * 2  # Since we're combining two pipelines
#         self.fc_combined = nn.Linear(combined_input_size, hidden_sizes[2])
#         self.relu_combined = nn.ReLU()
#         self.dropout_combined = nn.Dropout(dropout_rate)

#         # Output layer
#         self.fc_out = nn.Linear(hidden_sizes[2], output_size)
        
#     def forward(self, code_x, repo_x):
#         # Reformat input
#         # [batch_size, sequence_length, embedding_dim] -> [batch_size, embedding_dim, sequence_length, 1]
#         code_x = code_x.permute(0, 2, 1).unsqueeze(-1)
        
#         # Process code embeddings (2D input)
#         code_x = self.code_conv1(code_x)
#         code_x = self.code_relu1(code_x)
#         code_x = self.code_dropout1(code_x)
        
#         code_x = self.code_conv2(code_x)
#         code_x = self.code_relu2(code_x)
#         code_x = self.code_dropout2(code_x)
        
#         # Flatten the output from the conv layers before the fully connected layer
#         code_x = self.code_flatten(code_x)
#         code_x = self.code_fc(code_x)
        
#         # Process repo names (1D input)
#         repo_x = self.repo_fc1(repo_x)
#         repo_x = self.repo_relu1(repo_x)
#         repo_x = self.repo_dropout1(repo_x)
        
#         repo_x = self.repo_fc2(repo_x)
#         repo_x = self.repo_relu2(repo_x)
#         repo_x = self.repo_dropout2(repo_x)
        
#         # Combine both outputs
#         combined_x = torch.cat((code_x, repo_x), dim=1)
        
#         # Process the combined output
#         combined_x = self.fc_combined(combined_x)
#         combined_x = self.relu_combined(combined_x)
#         combined_x = self.dropout_combined(combined_x)
        
#         # Final output
#         output = self.fc_out(combined_x)
#         return output

In [ ]:
import torch.optim as optim
# Define model parameters

code_embedding_dim = df_train['embedding'][5].shape[0]

repo_input_size = len(df_train['repo_name_tensor'][0])
hidden_sizes = [1024, 512, 256, 128]
hidden_sizes_pipeline_1 = [1024, 512, 256]
hidden_sizes_pipeline_2 = [512, 256, 256, 128]
output_size = resulting_label_count
dropout_rate = 0.2

device = "cuda:0"
model = CodeDiffNN(code_embedding_dim, repo_input_size, hidden_sizes_pipeline_1,hidden_sizes_pipeline_2, output_size, dropout_rate).to(device)
# model = SinglePipelineCodeDiffNN(code_embedding_dim, hidden_sizes, output_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Decrease learning rate automatically on plateau
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=400, factor=0.1, verbose=True)

In [ ]:
print(resulting_label_count)
print(code_embedding_dim)
torch.cuda.empty_cache()

3
768


In [ ]:
# Function to calculate accuracy
def calculate_accuracy(outputs, labels):
    _, predicted = torch.max(outputs, 1)  # Get the index of the max log-probability
    # labels_1d = torch.argmax(labels, dim=1) # turn into 1D tensor
    correct = (predicted == labels).sum().item()
    accuracy = correct / labels.size(0)  # Accuracy = number of correct predictions / total number of samples
    return accuracy

In [ ]:
import tqdm

def train_with_data_loader(model,loader):
    
    running_loss = 0.0
    running_accuracy = 0.0
    
    for batch_X, batch_y, batch_X_repo, idx in tqdm.tqdm_notebook(loader):
        
        batch_X = scaler.transform(batch_X)
        batch_X = torch.tensor(batch_X,dtype=torch.float32)
        
        # Forward pass        
        batch_y = batch_y.to(device)
        batch_X = batch_X.to(device)
        batch_X_repo = batch_X_repo.to(device)
        
        batch_y = torch.argmax(batch_y, dim=1)

        # outputs = model(batch_X)
        outputs = model(batch_X, batch_X_repo)
        loss = criterion(outputs, batch_y)
    
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        # Calculate accuracy
        accuracy = calculate_accuracy(outputs, batch_y)
        
        # Update running loss and accuracy
        running_loss += loss.item()
        running_accuracy += accuracy
        
        
    val_loss = running_loss / len(loader)
    val_accuracy = running_accuracy / len(loader)
    
    return val_loss, val_accuracy

In [ ]:
def evaluate_with_data_loader(model,loader):
    
    running_loss = 0.0
    running_accuracy = 0.0
    
    for batch_X, batch_y, batch_X_repo, idx in loader:
        
        batch_X = scaler.transform(batch_X)
        batch_X = torch.tensor(batch_X,dtype=torch.float32)
        
        # Forward pass
        batch_y = batch_y.to(device)
        batch_X = batch_X.to(device)
        batch_X_repo = batch_X_repo.to(device)
        
        batch_y = torch.argmax(batch_y, dim=1)

        outputs = model(batch_X, batch_X_repo)
        # outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Calculate accuracy
        accuracy = calculate_accuracy(outputs, batch_y)
        
        # Update running loss and accuracy
        running_loss += loss.item()
        running_accuracy += accuracy
        
        
    val_loss = running_loss / len(loader)
    val_accuracy = running_accuracy / len(loader)
    
    # Adjust learning rate based on validation loss
    scheduler.step(val_loss)
    
    return val_loss, val_accuracy

In [ ]:
def test_with_data_loader(model,loader):
    
    running_loss = 0.0
    running_accuracy = 0.0
    predicted_y = []
    y = []
    
    for batch_X, batch_y, batch_X_repo, idx in loader:
        
        batch_X = scaler.transform(batch_X)
        batch_X = torch.tensor(batch_X,dtype=torch.float32)

        # Forward pass
        batch_y = batch_y.to(device)
        batch_X = batch_X.to(device)
        batch_X_repo = batch_X_repo.to(device)
        
        # index encoding
        batch_y_1D = torch.argmax(batch_y, dim=1)

        outputs = model(batch_X, batch_X_repo)
        # outputs = model(batch_X)
        loss = criterion(outputs, batch_y_1D)
        
        # Calculate accuracy
        accuracy = calculate_accuracy(outputs, batch_y_1D)
        
        # Update running loss and accuracy
        running_loss += loss.item()
        running_accuracy += accuracy

        predicted_y.append(outputs)
        y.append(batch_y)
        
    test_loss = running_loss / len(loader)
    test_accuracy = running_accuracy / len(loader)
    
    return test_loss, test_accuracy, torch.cat(predicted_y, dim=0), torch.cat(y, dim=0)

In [ ]:
import copy
import tqdm

# Train the model

# Initialize variables to track the best model and best accuracy
best_accuracy = 0.0
best_model = None

num_epochs = 500

for epoch in tqdm.tqdm_notebook(range(num_epochs)):
    model.train()  # Set the model to training mode
    
    # Print average loss and accuracy for the epoch
    avg_loss,avg_accuracy = train_with_data_loader(model= model,loader= train_loader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {avg_accuracy:.4f}')

    # Evaluate on validation data
    model.eval()  # Set the model to evaluation mode
    
    with torch.no_grad():  # Disable gradient computation for validation
        val_loss,val_accuracy = evaluate_with_data_loader(model=model,loader=val_loader)
        print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}')
    
    # Check if the current model has the best validation accuracy
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_model = copy.deepcopy(model.state_dict())  # Save the model with the best accuracy


C:\Users\FSM\AppData\Local\Temp\ipykernel_51976\902125487.py:12: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for epoch in tqdm.tqdm_notebook(range(num_epochs)):


  0%|          | 0/500 [00:00<?, ?it/s]

C:\Users\FSM\AppData\Local\Temp\ipykernel_51976\2221030542.py:8: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for batch_X, batch_y, batch_X_repo, idx in tqdm.tqdm_notebook(loader):


  0%|          | 0/252 [00:00<?, ?it/s]

Epoch [1/500], Loss: 0.9352, Accuracy: 0.6011
Validation Loss: 0.8220, Validation Accuracy: 0.6305


...

Epoch [499/500], Loss: 0.0604, Accuracy: 0.9789
Validation Loss: 2.6440, Validation Accuracy: 0.8100


C:\Users\FSM\AppData\Local\Temp\ipykernel_51976\2221030542.py:8: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for batch_X, batch_y, batch_X_repo, idx in tqdm.tqdm_notebook(loader):


  0%|          | 0/252 [00:00<?, ?it/s]

Epoch [500/500], Loss: 0.0860, Accuracy: 0.9794
Validation Loss: 2.6843, Validation Accuracy: 0.8092


: 

In [ ]:
# Save best model to file
model.load_state_dict(best_model)
torch.save(model.state_dict(), 'models/dynamic_lr_2_pipelines_StScaler_pooler_6layers_400epoch.pth')

In [ ]:
# Load model from file
model = CodeDiffNN(code_embedding_dim, repo_input_size, hidden_sizes_pipeline_1,hidden_sizes_pipeline_2, output_size).to(device)
model.load_state_dict(torch.load('models/fixed_model_2_pipelines_400epoch.pth'))

In [20]:
with torch.no_grad():
    # Load best model from previous training cycle
    model.load_state_dict(best_model)
    print(f'Best model accuracy on validation data: {best_accuracy:.4f}')

    model.eval()
    test_loss,test_accuracy,test_outputs,y_test = test_with_data_loader(model=model,loader=test_loader)
    
    print(f'Accuracy on test data: {test_accuracy:.4f}')
    print(test_outputs.shape)

Best model accuracy on validation data: 0.8133
Accuracy on test data: 0.8024
torch.Size([6885, 3])


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

_, predicted = torch.max(test_outputs.detach().cpu(), 1)  # Get the index of the max log-probability
labels_1d = torch.argmax(y_test, dim=1).cpu() # turn into 1D tensor

# Convert to numpy arrays
y_test_np = labels_1d.numpy()
predicted_np = predicted.numpy()

# Compute confusion matrix
cm = confusion_matrix(y_test_np, predicted_np)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title('Confusion Matrix for Multi-Class Classification')
plt.show()

print(predicted_np)
print(y_test_np)

In [ ]:
import gc
# Free up GPU memory (testing)
print(f"Cached GPU memory before: {torch.cuda.memory_reserved()} bytes")
del model
del best_model
del test_outputs
del y_test
del predicted
del labels_1d
del val_loader, test_loader, train_loader
del cm
gc.collect()
torch.cuda.empty_cache()
print(f"Cached GPU memory after: {torch.cuda.memory_reserved()} bytes")